In [ ]:
!pip install open_clip_torch

In [ ]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torch.nn as nn

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [ ]:
from clip_zeroshot import build_and_cache_text_features, build_and_cache_image_features, top_k_accuracy, load_cached_features

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

# Test the extracted coop.py

In [ ]:
from coop import PromptLearner, TextEncoderWrapper

### Download and prepare the caltech data

In [ ]:
caltech_dataset = datasets.Caltech101(root='./data', download=True, transform=preprocess)

In [ ]:
len(caltech_dataset)

In [ ]:
caltech_classes = [cls.replace('_', ' ') for cls in caltech_dataset.categories]
print(caltech_classes[:5])

In [ ]:
from collections import defaultdict

class_to_indices = defaultdict(list)

for idx in range(len(caltech_dataset)):
  label = caltech_dataset[idx][1]
  class_to_indices[label].append(idx)

In [ ]:
len(class_to_indices)

In [ ]:
import random

few_shot_indices = []

for label, indices in class_to_indices.items():
  sampled = random.sample(indices, 16)
  few_shot_indices.extend(sampled)

In [ ]:
few_shot_indices[:5]

In [ ]:
from torch.utils.data import Subset

few_shot_dataset = Subset(caltech_dataset, few_shot_indices)

In [ ]:
all_indices = set(range(len(caltech_dataset)))
eval_indices = list(all_indices - set(few_shot_indices))
eval_dataset = Subset(caltech_dataset, eval_indices)

In [ ]:
print(len(eval_dataset))
print(len(few_shot_dataset))

In [ ]:
raw_few_shot_loader = DataLoader(few_shot_dataset, batch_size=32, shuffle=True)
raw_eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

### Build the text features

In [ ]:
prompt_learner = PromptLearner(clip_model=model, device=device, n_ctx=4, tokenizer=tokenizer, ctx_dim=512, class_names=caltech_classes).to(device)
prompt, tok_prompt = prompt_learner()

In [ ]:
text_encoder = TextEncoderWrapper(model)
text_features = text_encoder(prompt, tok_prompt)

### Build the image features

In [ ]:
cache_img_train = build_and_cache_image_features(model, device, raw_few_shot_loader, './features', 'caltech_img_train')

In [ ]:
from torch.utils.data import TensorDataset

img_features_dataset = TensorDataset(cache_img_train['image_features'], cache_img_train['labels'])
train_loader = DataLoader(img_features_dataset, batch_size=32, shuffle=True)

### Training Loop

In [ ]:
for param in model.parameters():
  param.requires_grad_(False)

In [ ]:
from tqdm.notebook import tqdm
import torch.nn.functional as F

epochs = 10
optimizer = torch.optim.Adam(prompt_learner.parameters(), lr=0.002)
num_ctx = 4
ctx_dim = 312
logit_scale = model.logit_scale.exp()

for epoch in range(epochs+1):
  total_loss = 0
  for img_feat, labels in tqdm(train_loader):
    img_feat = img_feat.to(device)
    labels = labels.to(device)

    prompts, tok_prompts = prompt_learner()
    text_features = text_encoder(prompts, tok_prompt)
    text_features = text_features / text_features.norm(dim=-1,keepdim=True)

    logits = logit_scale * img_feat @ text_features.t()
    loss = F.cross_entropy(logits, labels)
    total_loss += loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  epoch_loss = total_loss / len(train_loader)
  print(f"epoch : {epoch + 1}, loss : {epoch_loss: .4f}")

### Evaluation

In [ ]:
eval_cached = build_and_cache_image_features(model, device, raw_eval_loader, './features', "caltech_eval")

eval_feature_dataset = TensorDataset(eval_cached["image_features"], eval_cached["labels"])
test_loader = DataLoader(eval_feature_dataset, batch_size=32, shuffle=False)

In [ ]:
prompt_learner.eval()
with torch.no_grad():
    prompts, tokenized = prompt_learner()
    text_features = text_encoder(prompts, tokenized)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    correct = 0
    total = 0
    for image_features, labels in test_loader:
        image_features = image_features.to(device)
        labels = labels.to(device)

        logits = image_features @ text_features.t()
        preds = logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test accuracy: {accuracy:.2f}")

# History

### Download and prepare the caltech data

In [ ]:
caltech_dataset = datasets.Caltech101(
    root='./data',
    download=True,
    transform=preprocess
)

In [ ]:
len(caltech_dataset)

In [ ]:
print(caltech_dataset.categories[:50])

In [ ]:
# Prepare the class names
caltech_class_names = [cls.replace('_', ' ') for cls in caltech_dataset.categories]
print(caltech_class_names[:10])

In [ ]:
# Create a dictionary of {class: [index_of_item1, index_of_item2....]}
from collections import defaultdict

total_labels = set()
class_to_indices = defaultdict(list)

for idx in range(len(caltech_dataset)):
    _, label = caltech_dataset[idx]
    class_to_indices[label].append(idx)
    total_labels.add(label)

In [ ]:
# Sample 16 images for every class
import random
random.seed(42)

few_shot_indices = []

for label, indices in class_to_indices.items():
   sampled = random.sample(indices, min(16, len(indices)))
   few_shot_indices.extend(sampled)

In [ ]:
# Build the training set
from torch.utils.data import Subset

few_shot_training_dataset = Subset(caltech_dataset, few_shot_indices)

In [ ]:
# Build the evaluation set
all_indices = set(range(len(caltech_dataset)))
eval_indices = list(all_indices - set(few_shot_indices))
eval_dataset = Subset(caltech_dataset, eval_indices)

In [ ]:
print(len(few_shot_training_dataset))
print(len(eval_dataset))

In [ ]:
raw_train_loader = DataLoader(few_shot_training_dataset, batch_size=32, shuffle = True)

### Build the prompt learner

In [ ]:
tokens = tokenizer(["a photo of a dog"]).to(device)
print(tokens.shape)   # confirm: token IDs, e.g. (1, 77)

embedded = model.token_embedding(tokens)   # try the layer you found
print(embedded.shape)  # should be (1, 77, embed_dim) — e.g. (1, 77, 512)

In [ ]:
class PromptLearner(nn.Module):
  def __init__(self, clip_model, n_ctx, tokenizer, ctx_dim, class_names):
    super().__init__()
    placeholder = "X " * n_ctx
    prompts = [f"{placeholder}{name}." for name in class_names]
    tokenized_prompts = tokenizer(prompts).to(device)
    self.num_classes = len(class_names)
    with torch.no_grad():
      embedding = clip_model.token_embedding(tokenized_prompts)

    prefix = embedding[:, :1, :]
    suffix = embedding[:, 1 + n_ctx:, :]

    self.register_buffer("prefix", prefix)
    self.register_buffer("suffix", suffix)
    self.register_buffer("tokenized_prompts", tokenized_prompts)
    self.ctx = nn.Parameter(torch.randn(n_ctx, ctx_dim) * 0.02)

  def forward(self):
    ctx = self.ctx.unsqueeze(0).expand(self.num_classes, -1, -1)
    prompts = torch.cat([self.prefix, ctx, self.suffix], dim=1)
    return prompts, self.tokenized_prompts

In [ ]:
pl = PromptLearner(model, 4, tokenizer, 512, caltech_class_names).to(device)
prompts, tok_prompts = pl()

In [ ]:
prompts.shape

### Build the TextEncoderWrapper

In [ ]:
from open_clip.transformer import text_global_pool

class TextEncoderWrapper(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.transformer = clip_model.transformer
        self.positional_embedding = clip_model.positional_embedding
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        self.attn_mask = clip_model.attn_mask
        self.text_pool_type = clip_model.text_pool_type
        self.text_eos_id = getattr(clip_model, "text_eos_id", None)

    def forward(self, prompt_embeddings, tokenized_prompts):
        cast_dtype = self.transformer.get_cast_dtype()
        x = prompt_embeddings.to(cast_dtype) + self.positional_embedding.to(cast_dtype)
        x = self.transformer(x, attn_mask=self.attn_mask)
        x = self.ln_final(x)
        x = text_global_pool(x, tokenized_prompts, self.text_pool_type, eos_token_id=self.text_eos_id)
        if self.text_projection is not None:
            if isinstance(self.text_projection, nn.Linear):
                x = self.text_projection(x)
            else:
                x = x @ self.text_projection
        return x

In [ ]:
text_encoder = TextEncoderWrapper(model)

text_features = text_encoder(prompts, tok_prompts)

In [ ]:
text_features.shape

### Build the image features

In [ ]:
train_image_cached = build_and_cache_image_features(model, device, raw_train_loader, './features', 'caltech-101-image')

In [ ]:
from torch.utils.data import TensorDataset

image_feature_dataset = TensorDataset(train_image_cached["image_features"], train_image_cached["labels"])
train_loader = DataLoader(image_feature_dataset, batch_size=32, shuffle=True)

### Training Loop

In [ ]:
for param in model.parameters():
    param.requires_grad_(False)

# PromptLearner's ctx should be the ONLY trainable thing
trainable = [name for name, p in pl.named_parameters() if p.requires_grad]
print(trainable)                    # expect just ['ctx']
print(pl.ctx.requires_grad)   # True

In [ ]:
from tqdm.notebook import tqdm
import torch.nn.functional as F

epochs = 10
optimizer = torch.optim.SGD(pl.parameters(), lr=0.002)
num_ctx = 4
ctx_dim = 512
logit_scale = model.logit_scale.exp()

for epoch in range(epochs + 1):
  total_loss = 0
  for img_features, labels in tqdm(train_loader):
    img_features = img_features.to(device)
    labels = labels.to(device)

    prompts, tok_prompts = pl()
    text_features = text_encoder(prompts, tok_prompts)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    logits = logit_scale * img_features @ text_features.t()
    loss = F.cross_entropy(logits, labels)
    total_loss += loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  epoch_loss = total_loss / len(train_loader)
  print(f"epoch: {epoch + 1}, loss: {epoch_loss:.4f}")

### Evaluation
Test accuracy: 90.21

In [ ]:
eval_raw_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

eval_cached = build_and_cache_image_features(model, device, eval_raw_loader, './features', "caltech_eval")

eval_feature_dataset = TensorDataset(eval_cached["image_features"], eval_cached["labels"])
test_loader = DataLoader(eval_feature_dataset, batch_size=32, shuffle=False)

In [ ]:
print([n for n, p in pl.named_parameters() if p.requires_grad])  # ['ctx'] only
print(sum(p.requires_grad for p in model.parameters()))  # 0

In [ ]:
pl.eval()
with torch.no_grad():
    # 1. get the TRAINED text features — ctx is now learned
    prompts, tokenized = pl()
    text_features = text_encoder(prompts, tokenized)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # 2. eval over the TEST set's cached image features
    correct = 0
    total = 0
    for image_features, labels in test_loader:   # cached test features
        image_features = image_features.to(device)
        labels = labels.to(device)

        logits = image_features @ text_features.t()
        preds = logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test accuracy: {accuracy:.2f}")

### Test Zero Shot CLIP
Zero-shot test accuracy: 84.58

In [ ]:
clip_text_features = build_and_cache_text_features(model=model, tokenizer=tokenizer, classnames=caltech_class_names, templates=['a photo of a {}.'], device=device, cache_dir='./features', file_name='caltech_text_features_clip')

In [ ]:
clip_text_features = clip_text_features.to(device)

In [ ]:
clip_text_features = clip_text_features.to(device)
clip_image_features = eval_cached['image_features'].to(device)
clip_labels = eval_cached['labels'].to(device)

correct = 0
total = 0

similarity = clip_image_features @ clip_text_features
preds = similarity.argmax(dim=-1)
correct += (clip_labels == preds).sum().item()
total += clip_labels.size(0)

accuracy = 100 * correct / total
print(f"Zero-shot test accuracy: {accuracy:.2f}")